# Analiza streszczeń literatury młodzieżowej

### import bibliotek

In [6]:
import os
import json
import string
import morfeusz2
import pandas as pd
import numpy as np
import operator as op 
import itertools as it
from wordcloud import WordCloud
from matplotlib import pyplot as plt
from nltk.util import ngrams
from nltk.tokenize import word_tokenize
from nltk.corpus import PlaintextCorpusReader
from scipy.cluster.hierarchy import dendrogram
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [7]:
import nltk
if "." not in nltk.data.path:
    nltk.data.path.append(".")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\s-A115-13\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Funkcje pomocnicze

In [8]:
def print_dict(obj):
    print(json.dumps(obj, indent=4, ensure_ascii=False))

In [9]:
def create_dir(path):
    if not os.path.exists(path):
        os.mkdir(path)

## Utworzenie korpusu dokumentów

In [15]:
corpus_dir= "./Literature"
corpus = PlaintextCorpusReader(corpus_dir, r".*\.txt")
files_names = corpus.fileids()
files_names

['Harry Potter i Czara Ognia.txt',
 'Harry Potter i Insygnia Smierci.txt',
 'Harry Potter i Kamien Filozoficzny.txt',
 'Harry Potter i Komnata Tajemnic.txt',
 'Harry Potter i Ksiaze Polkrwi.txt',
 'Harry Potter i Przeklete Dziecko.txt',
 'Harry Potter i Wiezien Azkabanu.txt',
 'Harry Potter i Zakon Feniksa.txt',
 'Opowiesci z Narni. Kon i jego chlopiec.txt',
 'Opowiesci z Narni. Ksiaze Kaspian.txt',
 'Opowiesci z Narni. Lew, czarownica i stara szafa.txt',
 'Opowiesci z Narni. Ostatnia bitwa.txt',
 'Opowiesci z Narni. Podroz Wedrowca do Switu.txt',
 'Opowiesci z Narni. Siostrzeniec Czarodzieja.txt',
 'Opowiesci z Narni. Srebrne krzeslo.txt',
 'Zmierzch.Ksiezyc w Nowiu.txt',
 'Zmierzch.Przed Switem.txt',
 'Zmierzch.Zacmienie.txt',
 'Zmierzch.Zmierzch.txt',
 'Zmierzch.Zycie i Smierc.txt']

In [16]:
documents_dict = {}
for file_name in files_names:
    documents_dict[file_name] = corpus.raw(file_name)
print_dict(documents_dict)

{
    "Harry Potter i Czara Ognia.txt": "Lord Voldemort, wraz ze swoim sługą Glizdogonem przybywa do domu należącego niegdyś do rodziny Riddle'ów położonego w Little Hangleton. Tam układają plan dotyczący zamordowania Harry'ego Pottera, co ma pomóc Czarnemu Panu w odzyskaniu jego dawnej mocy. Jest także mowa o wiernym słudze Czarnego Pana, który wkrótce ma znaleźć się w Hogwarcie. Wszystko to słyszy mieszkający nieopodal ogrodnik, Frank Bryce. Voldemort odkrywając, iż mężczyzna podsłuchuje, zaprasza go do środka i uśmierca. Harry budzi się z przerażającego snu. Jego blizna pulsuje okropnym bólem. Próbuje sobie przypomnieć o czym śnił, a z pamięci wyłania się obraz mrocznego pokoju, węża leżącego przed kominkiem oraz Glizdogona i Lorda Voldemorta planujących morderstwo. Harry zastanawia się czy powinien komuś powiedzieć o problemach z blizną, ponieważ, gdy bolała go ostatnim razem, Voldemort wdarł się do Hogwartu. Ostatecznie postanawia napisać do Syriusza i zejść na śniadanie. W trakci

In [17]:
documents_lenghts = {}
for file_name in files_names:
    documents_lenghts[file_name] = {
        "pre": len(word_tokenize(documents_dict[file_name], language="polish"))
    }
    print_dict(documents_lenghts)

{
    "Harry Potter i Czara Ognia.txt": {
        "pre": 6749
    }
}
{
    "Harry Potter i Czara Ognia.txt": {
        "pre": 6749
    },
    "Harry Potter i Insygnia Smierci.txt": {
        "pre": 2993
    }
}
{
    "Harry Potter i Czara Ognia.txt": {
        "pre": 6749
    },
    "Harry Potter i Insygnia Smierci.txt": {
        "pre": 2993
    },
    "Harry Potter i Kamien Filozoficzny.txt": {
        "pre": 2236
    }
}
{
    "Harry Potter i Czara Ognia.txt": {
        "pre": 6749
    },
    "Harry Potter i Insygnia Smierci.txt": {
        "pre": 2993
    },
    "Harry Potter i Kamien Filozoficzny.txt": {
        "pre": 2236
    },
    "Harry Potter i Komnata Tajemnic.txt": {
        "pre": 1610
    }
}
{
    "Harry Potter i Czara Ognia.txt": {
        "pre": 6749
    },
    "Harry Potter i Insygnia Smierci.txt": {
        "pre": 2993
    },
    "Harry Potter i Kamien Filozoficzny.txt": {
        "pre": 2236
    },
    "Harry Potter i Komnata Tajemnic.txt": {
        "pre": 1610
 

## Utworzenie macierzy częstości przed wstępnym przygotowaniem dokumentów

In [22]:
documents_df_pre = pd.DataFrame.from_dict(documents_dict, orient="index")
documents_df_pre.columns = ["content"]
display(documents_df_pre)

,content
Harry Potter i Czara Ognia.txt,"Lord Voldemort, wraz ze swoim sługą Glizdogone..."
Harry Potter i Insygnia Smierci.txt,"Podczas zebrania śmierciożerców, które odbywa ..."
Harry Potter i Kamien Filozoficzny.txt,Na ulicy Privet Drive pojawia się stary człowi...
Harry Potter i Komnata Tajemnic.txt,"Wuj Vernon jest zły, że Hedwiga robi bałagan w..."
Harry Potter i Ksiaze Polkrwi.txt,Mugolskiego premiera odwiedza Korneliusz Knot ...
Harry Potter i Przeklete Dziecko.txt,"Scena pierwsza oraz druga opowiada historię, k..."
Harry Potter i Wiezien Azkabanu.txt,Harry leży pod kocami w łóżku i odrabia prace ...
Harry Potter i Zakon Feniksa.txt,Dudley jest zły i wyżywa się na Harrym. Dudley...
Opowiesci z Narni. Kon i jego chlopiec.txt,Akcja książki rozgrywa się w Narnii za czasów ...
Opowiesci z Narni. Ksiaze Kaspian.txt,"Mija rok od wydarzeń opisanych w powieści ""Lew..."


In [23]:
cv = CountVectorizer()
tv = TfidfVectorizer()


In [26]:
matrix_tf = cv.fit_transform(documents_df_pre["content"])
sparsity_tf = 1 - (matrix_tf.nnz / (matrix_tf.shape[0] * matrix_tf.shape[1]))

In [27]:
matrices_parameters = {
    "pre": {
        "words": matrix_tf.shape[1],
        "sparsity": sparsity_tf,
        }
}
print_dict(matrices_parameters)

{
    "pre": {
        "words": 7469,
        "sparsity": 0.9095595126522962
    }
}


## Wstępne przygotowanie dokumentów

In [28]:
stopwords_file = open("./stopwords_pl.txt", "r", encoding="UTF-8")
stopwords = stopwords_file.read().splitlines()
stopwords_file.close()
stopwords = stopwords[4:]
print (stopwords, end=",")

['vol', 'o.o.', 'mgr', 'godz', 'zł', 'www', 'pl', 'ul', 'tel', 'hab', 'prof', 'inż', 'dr', 'i', 'u', 'aby', 'albo', 'ale', 'ani', 'aż', 'bardzo', 'bez', 'bo', 'bowiem', 'by', 'byli', 'bym', 'był', 'była', 'było', 'były', 'być', 'będzie', 'będą', 'chce', 'choć', 'co', 'coraz', 'coś', 'czy', 'czyli', 'często', 'dla', 'do', 'gdy', 'gdyby', 'gdyż', 'gdzie', 'go', 'ich', 'im', 'inne', 'iż', 'ja', 'jak', 'jakie', 'jako', 'je', 'jednak', 'jednym', 'jedynie', 'jego', 'jej', 'jest', 'jeszcze', 'jeśli', 'jeżeli', 'już', 'ją', 'kiedy', 'kilku', 'kto', 'która', 'które', 'którego', 'której', 'który', 'których', 'którym', 'którzy', 'lat', 'lecz', 'lub', 'ma', 'mają', 'mamy', 'mi', 'miał', 'mimo', 'mnie', 'mogą', 'może', 'można', 'mu', 'musi', 'na', 'nad', 'nam', 'nas', 'nawet', 'nic', 'nich', 'nie', 'niej', 'nim', 'niż', 'no', 'nowe', 'np', 'nr', 'o', 'od', 'ok', 'on', 'one', 'oraz', 'pan', 'po', 'pod', 'ponad', 'ponieważ', 'poza', 'przed', 'przede', 'przez', 'przy', 'raz', 'razie', 'roku', 'również

In [32]:
def lemmatize(text):
    morfer = morfeusz2.Morfeusz()
    segments = it.groupby(morfer.analyse(text), op.itemgetter(0,1))
    def dieambiguate(group):
        pairs = ((len(descr), lemma) for _, _, (_, lemma, descr, _, _,) in group)
        _, lemma = min(pairs)
        return lemma.split(":")[0]
    lemmas = (dieambiguate(group) for _, group in segments)
    return " ".join(filter(str.isalpha, lemmas))




In [34]:
for key in documents_dict:
    documents_dict[key] = documents_dict[key].lower()
    documents_dict[key] = " ".join([char for char in documents_dict[key] if char not in string.punctuation])
    documents_dict[key] = lemmatize(documents_dict[key])
    documents_dict[key] = " ".join([word for word in word_tokenize(documents_dict[key], language="polish") if word not in stopwords])
print_dict(documents_dict)

{
    "Harry Potter i Czara Ognia.txt": "litr r D V litr D e metr r tona r a e sekunda metr sekunda ł gram ą gram litr D gram n e metr r y bit y a D D metr n a litr e ż ą cent e gram n e gram D y ś D r D n y r D D litr e ó ł ż n e gram litr tona tona litr e godzina a n gram litr e tona n tona a metr k ł a D a ą litr a n D tona y cent ą cent y a metr r D a n a godzina a r r y e gram tona tona e r a cent metr a metr ó cent cent a r n e metr a n D y sekunda k a n e gram D a n e metr cent y e sekunda tona tona a k ż e metr a e r n y metr sekunda ł D e cent a r n e gram a n a k tona ó r y k r ó tona cent e metr a n a litr e ź ć sekunda ę godzina gram a r cent e sekunda y sekunda tona k tona sekunda ł y sekunda y metr e sekunda k a ą cent y n e D a litr gram r D n k f r a n k bit r y cent e V litr D e metr r tona D k r y a ą cent ż metr ę ż cent y n a D sekunda ł cent godzina e a r a sekunda a gram D ś r D k a ś metr e r cent a godzina a r r y bit D sekunda ę r e r a ż a ą cent e gram sekund